In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import re

In [ ]:
#loading dataset
df = pd.read_csv("spam.csv", encoding="latin-1")
df = df[["v1", "v2"]].rename(columns={"v1": "label", "v2": "text"})
df["label"] = df["label"].map({"ham": 0, "spam": 1})

print(f"Dataset shape: {df.shape}")
print(f"spam: {df['label'].sum()}, ham: {len(df) - df['label'].sum()}")
print("ham percentage")
print((len(df) - df['label'].sum())/len(df))
print("spam percentage")
print(df['label'].sum()/len(df))


Dataset shape: (5572, 2)
spam: 747, ham: 4825
ham percentage
0.8659368269921034
spam percentage
0.13406317300789664


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = text.strip()
    return text


df["clean_text"] = df["text"].apply(clean_text)

X = df["clean_text"]
y = df["label"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tfidf = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    print(f"\n--- {name} ---")
    print(f"Accuracy: {accuracy_score(y_te, preds):.4f}")
    print(classification_report(y_te, preds, target_names=["Ham", "Spam"]))
    return model


nb_model = evaluate("Naive Bayes", MultinomialNB(), X_train_tfidf, X_test_tfidf, y_train, y_test)
lr_model = evaluate("Logistic Regression", LogisticRegression(max_iter=1000), X_train_tfidf, X_test_tfidf, y_train, y_test)


--- Naive Bayes ---
Accuracy: 0.9695
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       965
        Spam       1.00      0.77      0.87       150

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115


--- Logistic Regression ---
Accuracy: 0.9570
              precision    recall  f1-score   support

         Ham       0.95      1.00      0.98       965
        Spam       0.98      0.69      0.81       150

    accuracy                           0.96      1115
   macro avg       0.97      0.85      0.89      1115
weighted avg       0.96      0.96      0.95      1115



In [ ]:
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

MAX_WORDS = 10000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_LEN)

model = Sequential([
    Embedding(MAX_WORDS, 64, input_length=MAX_LEN),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

model.fit(X_train_seq, y_train, epochs=5, batch_size=32, validation_split=0.1, verbose=1)

loss, acc = model.evaluate(X_test_seq, y_test, verbose=0)
print("\n lstm keras model")
print(f"Accuracy: {acc:.4f}")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 11s 59ms/step - accuracy: 0.9307 - loss: 0.2030 - val_accuracy: 0.9686 - val_loss: 0.0835
Epoch 2/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 11s 65ms/step - accuracy: 0.9908 - loss: 0.0362 - val_accuracy: 0.9686 - val_loss: 0.0850
Epoch 3/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 8s 67ms/step - accuracy: 0.9960 - loss: 0.0151 - val_accuracy: 0.9731 - val_loss: 0.0792
Epoch 4/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 7s 57ms/step - accuracy: 0.9975 - loss: 0.0080 - val_accuracy: 0.9731 - val_loss: 0.0965
Epoch 5/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 9s 68ms/step - accuracy: 0.9993 - loss: 0.0020 - val_accuracy: 0.9753 - val_loss: 0.1091

 lstm keras model
Accuracy: 0.9812
